# Understanding BOCPD Outputs

This notebook explains how to interpret the objects returned by ``BOCPD``:

- The **run-length posterior** :math:`P(r_t = r | x_{1:t})`.
- The **changepoint probability** :math:`P(r_t = 0 | x_{1:t})` (``cp_prob``).
- Derived summaries (MAP run length, expected run length, credible intervals, entropy).

Each section focuses on a specific quantity, shows how to compute it, and explains
when it is useful.


## 0. Environment setup (one-time)


In [1]:

import sys
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "fast_bocpd").exists() else CWD.parent
EXAMPLES_DIR = REPO_ROOT / "examples"

sys.path.append(str(REPO_ROOT))
sys.path.append(str(EXAMPLES_DIR))
print(f"Using repo root: {REPO_ROOT}")


Using repo root: /home/tiaan/Projects/Fast_BOCPD


## 1. Imports


In [2]:

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fast_bocpd import BOCPD, ConstantHazard, GaussianNIG

from _helpers import generate_piecewise_gaussian


## 2. Synthetic dataset

We create a time series with four regimes. ``changepoints`` contains the ground-truth
boundaries so we can compare against the detector's beliefs.


In [3]:

df, changepoints = generate_piecewise_gaussian(
    lengths=[90, 120, 80, 140],
    means=[0.0, 1.0, -0.3, 0.7],
    sigma=0.22,
    seed=42,
)
df.head()


,t,value,true_mean
0,0,0.109277,0.0
1,1,-0.030418,0.0
2,2,0.142491,0.0
3,3,0.335067,0.0
4,4,-0.051514,0.0


## 3. Configure BOCPD and run once


In [4]:

obs_model = GaussianNIG(mu0=0.0, kappa0=1.0, alpha0=0.05, beta0=0.01)
hazard = ConstantHazard(lambda_=130)
bocpd = BOCPD(obs_model, hazard, max_run_length=500)

posteriors = []
cp_probs = []
for value in df["value"]:
    posterior, cp_prob = bocpd.update(float(value))
    posteriors.append(posterior.copy())
    cp_probs.append(cp_prob)

posterior_matrix = np.vstack(posteriors)
weights = np.arange(posterior_matrix.shape[1])
map_run = np.argmax(posterior_matrix, axis=1)
expected_run = posterior_matrix @ weights
entropy = -(posterior_matrix * np.log(posterior_matrix + 1e-12)).sum(axis=1)
ci_low = np.array([np.argmax(p.cumsum() >= 0.05) for p in posterior_matrix])
ci_high = np.array([np.argmax(p.cumsum() >= 0.95) for p in posterior_matrix])


df["cp_prob"] = cp_probs
df["map_r"] = map_run
df["expected_r"] = expected_run
df["entropy"] = entropy
df["ci_low"] = ci_low
df["ci_high"] = ci_high

df.head()


,t,value,true_mean,cp_prob,map_r,expected_r,entropy,ci_low,ci_high
0,0,0.109277,0.0,0.007692,1,0.992308,0.045105,1,1
1,1,-0.030418,0.0,0.001243,2,1.989831,0.054610,2,2
2,2,0.142491,0.0,0.000885,3,2.988086,0.058568,3,3
3,3,0.335067,0.0,0.002028,4,3.978772,0.076036,4,4
4,4,-0.051514,0.0,0.001330,5,4.977187,0.076994,5,5


## 4. What is the run-length posterior?

The run-length posterior is a probability distribution over how long the current
regime has lasted. We can slice it at specific time steps to see how confident the
algorithm is.


In [5]:

def posterior_slice(t_index, title, r_max=120):
    fig = go.Figure(
        data=go.Bar(
            x=np.arange(r_max),
            y=posterior_matrix[t_index, :r_max],
            marker_color="#4E79A7",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Run length r",
        yaxis_title="Probability",
        template="plotly_white",
        height=340,
    )
    return fig

posterior_slice(85, "Posterior 5 steps before the first changepoint")


In [6]:

posterior_slice(215, "Posterior in an ambiguous region")


### Interpretation

- Sharp peaks indicate high confidence about the regime age. You can use MAP or expected
  run length directly in automation.
- Broad posteriors mean the detector is uncertain. Consider delaying alerts or combining
  multiple signals (e.g., entropy) when the posterior is diffuse.


## 5. Changepoint probability ``cp_prob`` (``posterior_r[0]``)

``cp_prob`` is the probability that the current timestep is a changepoint. Use it as a
score in threshold-based alerting.


In [7]:

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    specs=[[{}], [{"secondary_y": True}]],
)
fig.add_trace(go.Scatter(x=df["t"], y=df["value"], name="Observation", line=dict(color="#4E79A7", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df["t"], y=df["cp_prob"], name="P(changepoint)", line=dict(color="#F28E2B", width=2)), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df["t"], y=df["map_r"], name="MAP run length", line=dict(color="#E15759", width=2, dash="dash")), row=2, col=1, secondary_y=True)
for cp in changepoints:
    fig.add_vline(x=cp, line=dict(color="#888", dash="dash", width=1), opacity=0.4)
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_yaxes(title_text="P(CP)", range=[0, 1], row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="MAP run length", row=2, col=1, secondary_y=True)
fig.update_layout(title="Changepoint probability + MAP run length", height=560, template="plotly_white")
fig


**When to trigger alerts**

1. ``cp_prob >= 0.3`` and MAP run length resets (``<= 2``).
2. ``cp_prob`` stays above 0.2 for ``N`` consecutive samples.
3. Combine with entropy (next section) to avoid firing when the posterior is diffuse.


## 6. Run-length entropy

Entropy quantifies uncertainty: $H(r_t) = -\sum_r P(r_t=r) \log P(r_t=r)$.
High entropy indicates ambiguity even if ``cp_prob`` is moderate. Plotting entropy
helps explain “why an alert did/did not fire.”


In [8]:

entropy_fig = go.Figure()
entropy_fig.add_trace(go.Scatter(x=df["t"], y=df["entropy"], name="Entropy", line=dict(color="#59A14F", width=2)))
for cp in changepoints:
    entropy_fig.add_vline(x=cp, line=dict(color="#888", dash="dash", width=1), opacity=0.4)
entropy_fig.update_layout(title="Run-length entropy", xaxis_title="Time step", yaxis_title="Entropy", template="plotly_white", height=360)
entropy_fig


## 7. Credible interval + expected run length

The plot below overlays MAP (dashed), posterior mean (solid), and a 90% credible interval.
Use credible intervals to communicate uncertainty to stakeholders or to build
requirements like “raise alert only when 90% CI is below 5 samples.”


In [9]:

ci_fig = go.Figure()
ci_fig.add_trace(go.Scatter(x=df["t"], y=df["expected_r"], name="Expected r", line=dict(color="#4E79A7", width=2)))
ci_fig.add_trace(go.Scatter(x=df["t"], y=df["map_r"], name="MAP r", line=dict(color="#F28E2B", width=2, dash="dash")))
ci_fig.add_trace(
    go.Scatter(
        x=np.concatenate([df["t"], df["t"][::-1]]),
        y=np.concatenate([df["ci_low"], df["ci_high"][::-1]]),
        fill="toself",
        fillcolor="rgba(78,121,167,0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        name="90% credible interval",
        hoverinfo="skip",
    )
)
ci_fig.update_layout(title="Run-length summaries", xaxis_title="Time step", yaxis_title="Run length", template="plotly_white", height=420)
ci_fig


## 8. Example alert logic (probability + MAP + entropy)


In [10]:

THRESHOLD = 0.3
MAX_MAP = 2
ENTROPY_CAP = 2.5
COOLDOWN = 25
alerts = []
last_alert = -COOLDOWN
for idx, row in df.iterrows():
    if idx - last_alert < COOLDOWN:
        continue
    if row["cp_prob"] >= THRESHOLD and row["map_r"] <= MAX_MAP and row["entropy"] <= ENTROPY_CAP:
        alerts.append(row["t"])
        last_alert = idx
alerts


[np.float64(90.0)]

## 9. Summary

- ``posterior_r`` is the foundation. Inspect slices, entropy, or credible intervals when you need to explain model behaviour.
- ``cp_prob`` alone is often sufficient for simple alerting, but combining it with MAP run length or entropy reduces false positives.
- Expected run length communicates “how long the regime has lasted” and is useful for period-of-stability reporting.
- Credible intervals and entropy expose edge cases (e.g., when hazard is mis-specified).

Next: ``04_observation_models.ipynb`` (coming soon) covers the statistical differences between observation models.
